# Test cross-reservoir: LSTM + PVT-encoder (Norne → Volve)

Entrenamos un LSTM seq-to-seq con encoder de PVT sobre **Norne** (30 simulaciones con variaciones del schedule de inyección de agua) y lo evaluamos contra **Volve**, un campo del Mar del Norte que el modelo nunca vio. Ambos datasets tienen el mismo schema de 16 columnas (petrofísica + PVT + operativas + cumuladas + target).

**¿Por qué no hay target leakage?** El target es `Presion_Reservorio_psi` (Pr) en cada timestep. El modelo nunca ve Pr en el timestep que está prediciendo — solo ve la historia operativa (cumuladas y caudales), la geometría y petrofísica del reservorio (constantes en el tiempo), y las propiedades termodinámicas del fluido (tabla PVT). La presión inicial `P_init` se usa para normalizar el target (predecimos `(Pr - P_init) / 5000`), pero `P_init` es una condición de borde conocida desde antes del primer minuto de producción — no es el valor que queremos predecir.

**Receta de entrenamiento**: 30 epochs fijos sin early stopping, con cosine LR schedule de `T_max=30`. Esto sale de un análisis previo de la curva de overfitting: el modelo mejora rápido hasta epoch ~30 y luego empieza a memorizar Norne, perdiendo capacidad de transferir a Volve.

## 1. Setup

Imports, semillas fijas, paths del proyecto y declaración de las columnas que componen los features estáticos (constantes por simulación) y dinámicos (cambian por timestep). `PRESSURE_SCALE = 5000` define la unidad en la que el modelo opera con presiones: todas se dividen por este valor para que el delta target viva en un rango ~tanh-friendly cercano a [-1, 1].

In [34]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
torch.manual_seed(42); np.random.seed(42)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "datasets"

PRESSURE_SCALE = 5000.0          # delta = (Pr - P_init) / PRESSURE_SCALE
M3_TO_BBL = 6.28981
TARGET = "Presion_Reservorio_psi"

STATIC_COLS = ["Porosidad", "log10_Permeabilidad_mD", "P_initial_norm"]
DYNAMIC_COLS = ["Np_over_PV", "Wp_over_PV", "Winj_over_PV", "GOR_cum",
                "qo_over_PV", "qwinj_over_PV", "WOR_inst", "water_cut_cum", "VRR_simple"]


## 2. Datasets y feature engineering

Cargamos los dos datasets: **Norne** (30 sims, con variaciones solo en el schedule de inyección de agua) y **Volve** (10 sims, reservado exclusivamente para la evaluación cross-reservoir final).

A partir del schema crudo de 16 columnas (porosidad, permeabilidad, cumuladas, caudales, etc.) construimos **12 features adimensionales**. La adimensionalización es el truco que habilita el transfer cross-reservoir: dividimos las cumuladas (`Np`, `Wp`, `Winj`) por el pore-volume del reservorio y los caudales por el mismo pore-volume. Así el modelo no aprende _"este campo lleva Np > 1e8 bbl"_ sino _"este campo ya recuperó X% de su volumen total"_ — una cantidad comparable entre Norne (campo grande) y Volve (campo chico). Las features incluyen también GOR acumulado, WOR instantáneo, water-cut acumulado y un VRR (voidage replacement ratio) simplificado.

In [35]:
def build_features(df):
    """Construye 12 features adimensionales (3 estaticos + 9 dinamicos) desde el schema crudo de 16 columnas.

    Estaticos por simulacion: porosidad, log10 permeabilidad, presion inicial normalizada.
    Dinamicos por timestep: cumuladas (Np, Wp, Winj) divididas por el pore-volume,
    GOR acumulado, caudales instantaneos sobre pore-volume, WOR instantaneo, water cut
    acumulado y VRR simplificado. La normalizacion por pore-volume saca la dependencia
    del tamano fisico del reservorio y hace que las features sean comparables entre
    Norne y Volve — pilar del transfer cross-reservoir.
    """
    out = pd.DataFrame({c: df[c] for c in ["sim_id", "reservoir_id", "tiempo_dias", TARGET]})
    pv = df["Area"] * df["Espesor_Neto_m"] * df["Porosidad"] * M3_TO_BBL
    out["Porosidad"] = df["Porosidad"]
    out["log10_Permeabilidad_mD"] = np.log10(df["Permeabilidad_mD"].clip(lower=1e-3))
    out["P_initial_norm"] = df.groupby(["reservoir_id", "sim_id"])[TARGET].transform("first") / PRESSURE_SCALE
    out["Np_over_PV"] = df["Prod_Acumulada_Petroleo"] / pv
    out["Wp_over_PV"] = df["Prod_Acumulada_Agua"] / pv
    out["Winj_over_PV"] = df["Iny_Acumulada_Agua"] / pv
    with np.errstate(divide="ignore", invalid="ignore"):
        out["GOR_cum"] = np.where(df["Prod_Acumulada_Petroleo"] > 0,
                                  df["Prod_Acumulada_Gas"] / df["Prod_Acumulada_Petroleo"], 0.0)
    out["qo_over_PV"] = df["Caudal_Prod_Petroleo_bbl"] / pv
    out["qwinj_over_PV"] = df["Caudal_Iny_Agua_bbl"] / pv
    g = df.groupby(["reservoir_id", "sim_id"])
    wp_rate = g["Prod_Acumulada_Agua"].diff() / g["tiempo_dias"].diff().replace(0, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        out["WOR_inst"] = np.where(df["Caudal_Prod_Petroleo_bbl"] > 0,
                                    wp_rate / df["Caudal_Prod_Petroleo_bbl"], 0.0)
        total = df["Prod_Acumulada_Petroleo"] + df["Prod_Acumulada_Agua"]
        out["water_cut_cum"] = np.where(total > 0, df["Prod_Acumulada_Agua"] / total, 0.0)
        out["VRR_simple"] = np.where(total > 0, df["Iny_Acumulada_Agua"] / total, 0.0)
    return out.fillna(0.0)


norne_raw = pd.read_csv(DATA / "dataset_norne.csv")
volve_raw = pd.read_csv(DATA / "dataset_volve.csv")
norne_feat = build_features(norne_raw)
volve_feat = build_features(volve_raw)
print(f"Norne features: {norne_feat.shape}, sims={norne_feat['sim_id'].nunique()}")
print(f"Volve features: {volve_feat.shape}, sims={volve_feat['sim_id'].nunique()}")


Norne features: (10580, 16), sims=30
Volve features: (4144, 16), sims=10


## 3. Vectores PVT por reservorio

**¿Qué es la tabla PVT** (Pressure-Volume-Temperature)? Es la huella termodinámica del fluido del reservorio. Para cada nivel de presión contiene:

- `Bo` (rb/stb): cuántos barriles _de fondo_ ocupa un barril de petróleo medido _en superficie_.
- `Bg` (rb/scf): lo mismo para el gas.
- `Rs` (scf/stb): cuánto gas viene disuelto en cada barril de petróleo a esa presión.
- `Pb` (psi): _presión de burbuja_, por debajo de la cual el gas empieza a liberarse del petróleo (cambia fuerte la compresibilidad y la dinámica del flujo).

**¿Por qué el modelo necesita esta información?** La ecuación de balance de materiales (Material Balance Equation) que gobierna la evolución de `Pr` depende directamente de `Bo`, `Bg`, `Rs` y `Pb`. Dos reservorios con la misma producción acumulada pero distinto PVT tienen dinámicas de presión completamente distintas. Sin estas features el modelo no puede generalizar entre fluidos diferentes — la PVT es lo que le dice al modelo _"qué tipo de fluido tenés en frente"_.

**Encoding**: cada reservorio se representa con un vector de **52 dimensiones** que concatena `[Pb, Bo×17, Bg×17, Rs×17]` con las tres curvas sobre los 17 puntos de presión de la tabla. Norne y Volve traen sus respectivas tablas canónicas — ambas ya están definidas sobre el mismo grid de presión (1500-5500 psi, 17 puntos), así que no hace falta interpolar. Cada simulación hereda el vector PVT de su reservorio: las 30 sims de Norne comparten el PVT canónico de Norne, las 10 de Volve comparten el PVT canónico de Volve.

In [36]:
def build_pvt_vector(pvt_dict):
    """Construye el vector PVT de 52D que ve el encoder: [Pb_norm, Bo·17, Bg·17·1000, Rs·17/1000].

    Las curvas Bo, Bg, Rs vienen de la tabla del laboratorio (mediciones fisicas del fluido).
    Pb es la presion de burbuja escalar. El re-escalado por 1000 / 1/1000 hace que las
    tres curvas vivan en ordenes de magnitud comparables antes de entrar al encoder
    (Bo ~ O(1), Bg sin escalar ~ O(1e-3), Rs sin escalar ~ O(1e3)).
    """
    pb = float(pvt_dict["pb_psi"]) / PRESSURE_SCALE
    bo = np.asarray(pvt_dict["bo_rb_stb"])
    bg = np.asarray(pvt_dict["bg_rb_scf"]) * 1000.0
    rs = np.asarray(pvt_dict["rs_scf_stb"]) / 1000.0
    return np.concatenate([[pb], bo, bg, rs]).astype(np.float32)


pvt_by_reservoir = {
    "norne": build_pvt_vector(json.loads((DATA / "pvt_norne.json").read_text())),
    "volve": build_pvt_vector(json.loads((DATA / "pvt_volve.json").read_text())),
}
print(f"PVT dim: {len(pvt_by_reservoir['norne'])}, reservoirs: {list(pvt_by_reservoir)}")


PVT dim: 52, reservoirs: ['norne', 'volve']


## 4. Modelo

**Arquitectura**: LSTM seq-to-seq de 2 capas (hidden=64). El input principal son los 9 features dinámicos por timestep, pero además el modelo recibe dos vectores _contextuales_ que valen para toda la simulación: los 3 features estáticos y el vector PVT de 52 dimensiones.

**¿Por qué necesitamos un encoder para el PVT (y otro para los estáticos)?** El vector PVT vive en un espacio de 52 dimensiones, que es excesivo para concatenar directamente con los 9 features dinámicos en cada timestep. Si lo hiciéramos:

1. La señal del PVT quedaría diluida (52 dimensiones constantes vs 9 que cambian).
2. El LSTM tendría que aprender por sí solo cuáles de las 52 dimensiones son informativas para predecir `Pr`, gastando capacidad en lo que es una tarea de compresión casi estática.
3. La cantidad de parámetros del LSTM crecería bastante (input_size 61 vs 41), aumentando el riesgo de overfitting con un dataset chico.

Un MLP encoder hace ese trabajo de compresión por adelantado: aprende a destilar las 52 dimensiones a un embedding de 16D que captura solo lo útil para la dinámica de presión. Lo mismo con el encoder estático (3D → 16D). Los dos embeddings se concatenan en un contexto de 32D que se _broadcast-ea_ sobre el eje temporal antes de entrar al LSTM. La cabeza es un MLP pequeño que emite el delta target normalizado por timestep.

**Sobre la generalización del PVT-encoder**: durante el training todas las simulaciones comparten el mismo PVT (el canónico de Norne), así que el encoder ve un input constante y su contribución durante el entrenamiento es básicamente un offset aprendido. En inference, cuando Volve aparece con su propio vector PVT, el encoder lo mapea a un embedding distinto que el LSTM puede usar para condicionar sus predicciones al nuevo fluido.

In [37]:
class LSTMPVT(nn.Module):
    """LSTM seq-to-seq con dos MLP encoders (uno para PVT y otro para features estaticos) que aportan contexto por sim.

    Flow del forward: el PVT (52D) pasa por su encoder y sale como vector 16D; los
    estaticos (3D) hacen lo mismo. Los dos embeddings se concatenan (-> 32D) y se
    broadcastean a cada timestep, donde se concatenan con los features dinamicos (9D)
    antes de entrar al LSTM (input_size = 9 + 32 = 41, hidden = 64, 2 layers). La head
    es un MLP pequeno que emite el delta target normalizado (Pr - P_init) / PRESSURE_SCALE
    en cada timestep.
    """
    def __init__(self, n_dyn=9, n_static=3, pvt_dim=52, ctx_dim=16, hidden=64, num_layers=2, dropout=0.15):
        super().__init__()
        self.pvt_encoder = nn.Sequential(nn.Linear(pvt_dim, 32), nn.GELU(), nn.Linear(32, ctx_dim))
        self.static_encoder = nn.Sequential(nn.Linear(n_static, 16), nn.GELU(), nn.Linear(16, ctx_dim))
        self.lstm = nn.LSTM(n_dyn + 2 * ctx_dim, hidden, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.GELU(), nn.Linear(32, 1))

    def forward(self, X_dyn, X_static, pvt):
        ctx = torch.cat([self.pvt_encoder(pvt), self.static_encoder(X_static)], dim=-1)
        ctx = ctx.unsqueeze(1).expand(-1, X_dyn.shape[1], -1)
        out, _ = self.lstm(torch.cat([X_dyn, ctx], dim=-1))
        return self.head(out).squeeze(-1)


## 5. Split, tensorización y entrenamiento

**Splits**: las 30 sims de Norne se dividen en 24 train, 3 val, 3 test (proporción 80/10/10 por sim_id). Volve queda completamente fuera del training y se reserva para la sección 7.

**Z-score**: se ajustan dos `StandardScaler` (uno para dinámicas, otro para estáticas) **solo sobre train** — usar val/test/Volve para ajustar las medias y desvíos sería leakage. El scaler entrenado se aplica después a los otros splits.

**Padding**: las simulaciones tienen distinto número de timesteps. Usamos `torch.nn.utils.rnn.pad_sequence` para apilarlas en un tensor batched y una `mask` boolean para indicar qué timesteps son reales vs padding — la mask se multiplica al MSE para que el padding no contribuya a la loss.

**Entrenamiento**: 30 epochs fijos, optimizador AdamW (lr=1e-3, wd=1e-4) y `CosineAnnealingLR` con `T_max=30`. El cosine schedule baja la learning rate de 1e-3 a ~0 a lo largo del entrenamiento — actúa como _regularización temporal_ natural y previene la divergencia tardía que se observa en runs largos sin schedule.

In [38]:
TRAIN_SIMS = list(range(1, 25))
VAL_SIMS = [25, 26, 27]
TEST_SIMS = [28, 29, 30]

tr_df = norne_feat[norne_feat["sim_id"].isin(TRAIN_SIMS)]
va_df = norne_feat[norne_feat["sim_id"].isin(VAL_SIMS)]
te_df = norne_feat[norne_feat["sim_id"].isin(TEST_SIMS)]

scaler_dyn = StandardScaler().fit(tr_df[DYNAMIC_COLS])
scaler_stat = StandardScaler().fit(tr_df.groupby("sim_id").first()[STATIC_COLS])


def pack(df):
    """Empaqueta las secuencias por sim en tensores padded y devuelve toda la metadata necesaria para evaluar.

    Por cada sim:
      - normaliza los features dinamicos y estaticos con los scalers ya ajustados en train,
      - rescata el vector PVT del reservorio correspondiente desde `pvt_by_reservoir`,
      - calcula el delta target (Pr - P_init) / PRESSURE_SCALE timestep a timestep,
      - guarda el array de Pr crudo en `targets` para denormalizar despues en `score`.
    Todas las sims se padean al T_max del split via `pad_sequence`; la `mask` indica
    timesteps reales vs padding (se usa luego para enmascarar la loss).
    """
    groups = list(df.groupby("sim_id", sort=True))
    X_dyn = pad_sequence([torch.tensor(scaler_dyn.transform(g[DYNAMIC_COLS]), dtype=torch.float32)
                          for _, g in groups], batch_first=True)
    X_static = torch.tensor(scaler_stat.transform(np.stack([g[STATIC_COLS].iloc[0] for _, g in groups])),
                            dtype=torch.float32)
    pvt = torch.tensor(np.stack([pvt_by_reservoir[g["reservoir_id"].iloc[0]] for _, g in groups]))
    p_init = torch.tensor([g["P_initial_norm"].iloc[0] * PRESSURE_SCALE for _, g in groups],
                          dtype=torch.float32)
    targets = [g[TARGET].to_numpy() for _, g in groups]
    delta = pad_sequence([torch.tensor((y - p.item()) / PRESSURE_SCALE, dtype=torch.float32)
                          for y, p in zip(targets, p_init)], batch_first=True)
    lengths = torch.tensor([len(g) for _, g in groups])
    mask = (torch.arange(X_dyn.shape[1])[None, :] < lengths[:, None]).float()
    return {"X_dyn": X_dyn, "X_static": X_static, "pvt": pvt, "delta": delta,
            "p_init": p_init, "mask": mask, "targets": targets}


tr, va, te, vo = pack(tr_df), pack(va_df), pack(te_df), pack(volve_feat)

model = LSTMPVT(n_dyn=len(DYNAMIC_COLS), n_static=len(STATIC_COLS), pvt_dim=tr["pvt"].shape[1])
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
mse = nn.MSELoss(reduction="none")

for _ in range(30):
    model.train()
    loss = (mse(model(tr["X_dyn"], tr["X_static"], tr["pvt"]), tr["delta"]) * tr["mask"]).sum() / tr["mask"].sum()
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
model.eval()
print(f"final training delta-MSE: {loss.item():.5f}")


final training delta-MSE: 0.00029


## 6. Norne — validación y test

Métricas `R²` y `MAE` sobre las 3 sims de validación y las 3 sims de test (ambas dentro de la distribución Norne, fold-out por sim_id). Como referencia, un baseline _naive_ que siempre predice `P_init` da MAE ~80 psi sobre Norne (ahí el schedule no mueve la presión más que ~100-200 psi).

In [39]:
@torch.no_grad()
def score(split, name):
    """Corre el modelo sobre un split empaquetado, denormaliza delta -> psi y devuelve una Serie pandas con R² y MAE.

    El modelo emite el delta normalizado `(Pr - P_init) / PRESSURE_SCALE`; aca lo
    multiplicamos por PRESSURE_SCALE y le sumamos P_init para volver a psi. Se concatenan
    todas las sims (sin padding gracias a `targets[i]` que tiene la longitud real) y se
    calculan las metricas con sklearn.
    """
    delta = model(split["X_dyn"], split["X_static"], split["pvt"]).numpy()
    p_init = split["p_init"].numpy()
    y_pred = np.concatenate([p_init[i] + delta[i, :len(t)] * PRESSURE_SCALE
                             for i, t in enumerate(split["targets"])])
    y_true = np.concatenate(split["targets"])
    return pd.Series({"sims": len(split["targets"]), "rows": len(y_true),
                      "R²": round(r2_score(y_true, y_pred), 4),
                      "MAE (psi)": round(mean_absolute_error(y_true, y_pred), 1)},
                     name=name)


pd.DataFrame([score(va, "Norne val"), score(te, "Norne test")])


,sims,rows,R²,MAE (psi)
Norne val,3.0,1069.0,0.8906,55.1
Norne test,3.0,1067.0,0.8619,66.0


## 7. Volve — evaluación cross-reservoir

Volve nunca se vio durante el training. **Campo distinto, PVT distinto, geometría distinta, presión inicial distinta**. La pregunta clave del experimento es si el modelo aprendió la _física_ subyacente (cómo evoluciona `Pr` en función de extracción/inyección y propiedades del fluido) o si solo memorizó la distribución específica de Norne.

**Lo que carga el transfer**: los features adimensionales se encargan de las diferencias geométricas y de escala entre los dos campos; el PVT-encoder es lo que le dice al modelo _"este reservorio tiene otro fluido"_, permitiéndole condicionar sus predicciones al PVT específico de Volve aunque nunca lo haya visto.

Como referencia, un baseline _naive_ que siempre predice `P_init` da MAE ≈ 197 psi y R² ≈ +0.22 sobre Volve. Cualquier número significativamente mejor implica que el modelo realmente está modelando la depleción/repesurización del reservorio.

In [40]:
pd.DataFrame([score(vo, "Volve (cross-reservoir)")])

,sims,rows,R²,MAE (psi)
Volve (cross-reservoir),10.0,4144.0,0.7005,114.3
